# Блок 6. Практика: Визуализация и дашборды

В этом notebook:
- SQL-запросы для панелей Grafana
- Проверка работоспособности запросов
- Подготовка данных для визуализации

**Основная работа с Grafana выполняется в веб-интерфейсе (http://localhost:3000)**

In [ ]:
import duckdb

# Используем данные из предыдущих блоков
PARQUET_PATH = "../lesson03/data/auth_events.parquet"

# Проверяем данные
duckdb.sql(f"FROM '{PARQUET_PATH}' SELECT COUNT(*) as total").show()

## SQL-запросы для панелей Grafana

Эти запросы можно использовать в Grafana после замены макросов:
- `$__timeFilter(timestamp)` → `timestamp > NOW() - INTERVAL '24 hours'`
- `$__timeGroup(timestamp, '1h')` → `DATE_TRUNC('hour', timestamp)`

In [ ]:
# Time Series: события по времени (успешные vs неудачные)
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT 
        DATE_TRUNC('hour', timestamp) as time,
        SUM(CASE WHEN success THEN 1 ELSE 0 END) as successful,
        SUM(CASE WHEN NOT success THEN 1 ELSE 0 END) as failed
    GROUP BY ALL
    ORDER BY time
""").show()

In [ ]:
# Stat: общее количество неудачных попыток
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT COUNT(*) as failed_total
    WHERE success = false
""").show()

In [ ]:
# Bar Chart: TOP-10 IP по неудачным попыткам
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT source_ip, COUNT(*) as failed_count
    WHERE success = false
    GROUP BY ALL
    ORDER BY failed_count DESC
    LIMIT 10
""").show()

In [ ]:
# Table: последние подозрительные события
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT timestamp, username, source_ip, event_type
    WHERE success = false
    ORDER BY timestamp DESC
    LIMIT 20
""").show()

## Следующие шаги

1. Запустите инфраструктуру: `docker compose up -d` (из solutions/lesson01)
2. Откройте Grafana: http://localhost:3000 (admin / пароль из .env)
3. Добавьте PostgreSQL как Data Source
4. Создайте дашборд с панелями, используя запросы выше
5. Добавьте переменную для фильтрации по username

**Совет:** В Grafana используйте макросы `$__timeFilter()` и `$__timeGroup()` вместо явных дат.